In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [6]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.vectorstores import InMemoryVectorStore
from langchain.tools import tool
from langchain.agents import create_agent

In [3]:
loader=PyPDFLoader("../Data/DT_Gene_Fairoz.pdf")
docs=loader.load()

In [4]:
splitter=RecursiveCharacterTextSplitter(chunk_size=800,chunk_overlap=150)
splitted_docs=splitter.split_documents(docs)

In [5]:
embeddings=OllamaEmbeddings(model="nomic-embed-text")
vector_store=InMemoryVectorStore.from_documents(documents=splitted_docs,
                                                embedding=embeddings
                                                )


In [8]:
@tool 
def retreival_tool(query:str):
    """
    This tool should be used to retreive the context for a query.
    """
    retreived_chunks=vector_store.similarity_search(query)
    context=""
    for data in context:
        context+=data.page_content

    return context

In [9]:
llm=ChatOllama(model="gemma4:31b-cloud")

In [12]:
agent=create_agent(
    model=llm,
    tools=[retreival_tool],
    system_prompt="You are a helpful Assistant that helps with the Q&A Make sure you always use the 'retreival_tool' for obtaining context and then only answer"
)